# Reporting Bibliometrics Plots

This notebook demonstrates how to:

- Configure input parameters

- Retrieve bibliometric data by publication year

- Generate reporting visualizations, including:
    - Annual science publication counts by mission, comparing human vs. LLM outputs
    - Aggregated publication counts across multiple missions (human vs. LLM)
    - Overlay plots combining aggregated counts with individual mission trends

## Imports

In [ ]:
%matplotlib inline

from pathlib import Path

from bibcat.reporting.bibliometrics import FilterConfig, build_plot_data, load_records
from bibcat.reporting.plotting import (
    PlotStyle,
    apply_matplotlib_style,
    plot_overlay,
    plot_per_mission,
    plot_sum,
    plot_sum_excl_flagship,
)

## Input Data
Each entry in the input JSON file should follow the format shown in the example below. The minimum required keys are `publication_date`, `publisher_name`, `class_missions`, `scores`, and `missions`. `class_missions` and `missions` can be empty lists,`[]`.
 
```json
{
  "bibcode": "2024paper",
  "doi": "10.xxxx/xxxx-xxxx/xxxx",
  "title": "GALEX study",
  "publication_date": "2024-01-01T00:00:00",
  "publisher_name": "EDP",
  "class_missions": [
    {
      "mission": "GALEX",
      "paper_type": "SCIENCE"
    }
  ],
  "scores": [
    {
      "missions": {
        "GALEX": {
          "mission": "GALEX",
          "papertype": "MENTION"
        }
      }
    }
  ]
}
```

In [ ]:
# input data assignment
input_file = "/Users/jyoon/Documents/asb/bibliography_automation/historic_dataset/papertrack_classifications_and_scores_export_all_2026-02-25.json"

JSON_PATH = Path(input_file)

## User Configuration

You can specify the following parameters:

- **`missions`**: List of missions of interest. You may include all MAST missions, for example:
  ```txt
  ["BEFS", "Copernicus", "EUVE", "FIRST", "FUSE", "GALEX", "HST", "HUT", "IMAPS", "IUE", "JWST", "K2", "Kepler", "PanSTARRS", "Roman", "TESS", "TUES", "UIT", "WUPPE"]

- **`flagship_missions`**: If provided, these missions will be excluded from the aggregated plot of `missions`.

- **`paper_type`**: The paper classification type (e.g., `SCIENCE` or `MENTION`), available for both human and LLM classifications.

- **`start_year` and `end_year`**: Range of publication years to include.

- **`publisher_include` and `publisher_exclude`** *(optional)*:  
  Use these fields to filter publications by publisher.

  - Use `publisher_exclude` to omit specific publishers (e.g., to exclude IOP journals).
  - Use `publisher_include` to restrict results to a specific set of publishers.

  **Note:** As of March 18, 2026, IOP (AAS journals) data is not available. Excluding `"IOP"` will ensure only non-IOP publications are included.

    The currently available publishers in MAST include:

    ```txt
    ["AAAS", "APS", "Annual Reviews", "Cambridge UP", "EDP", "Elsevier", "Hindawi (OA)", "IOP", "Liebert", "MDPI (OA)", "Oxford UP", "SPIE", "SpringerNature", "Universidad Nacional Autónoma de México (UNAM) Press", "Wiley", "World Scientific"]

In [ ]:
MISSION_ACTIVE_RANGES = {
    "IUE": (1978, 1998),
    "EUVE": (1992, 2000),
    "FUSE": (1999, 2008),
    "GALEX": (2004, 2013),
    "PANSTARRS": (2009, 2015),
    "KEPLER": (2009, 2018),
    "K2": (2014, 2018),
    "TESS": (2018, 2025),
    "HST": (1990, 2025),
}

config = FilterConfig(
    missions=[
        "BEFS",
        "Copernicus",
        "EUVE",
        "FIRST",
        "FUSE",
        "GALEX",
        "HST",
        "HUT",
        "IMAPS",
        "IUE",
        "JWST",
        "K2",
        "Kepler",
        "PanSTARRS",
        "Roman",
        "TESS",
        "TUES",
        "UIT",
        "WUPPE",
    ],  # ["FUSE", "HST", "GALEX", "K2", "KEPLER", "PANSTARRS", "TESS"],
    flagship_missions=["JWST", "ROMAN", "HST"],
    paper_type="SCIENCE",
    start_year=1976,
    end_year=2025,
    publishers_include=["Oxford UP"],
    publishers_exclude=None,
    exclude_publishers_as_substring=True,
)

style = PlotStyle(save_figs=False)  # True if you want to save figures as .png

## Loading and Plotting Bibliometrics Data

Bibliometric data processing and visualization are handled by bibliometrics.py and plotting.py.
- bibliometrics.py is used to load and process the data
- plotting.py contains the plotting functions and style configurations, which can be modified as needed

In [ ]:
apply_matplotlib_style()

records = load_records(JSON_PATH)
data = build_plot_data(
    records=records,
    config=config,
    mission_active_ranges=MISSION_ACTIVE_RANGES,
)

In [ ]:
plot_sum(data, style)
plot_sum_excl_flagship(data, style)
plot_overlay(data, style, kind="human", sum_mode="all")
plot_overlay(data, style, kind="llm", sum_mode="excl_flagship")
plot_per_mission(data, style)